# Spectral Synthesis — documentation examples

Companion notebook to the **Spectral Synthesis** documentation page
(`docs/source/spectral_synthesis.rst`). One section per code snippet.

Requires only the core `stellar-spice` install.

In [1]:
%matplotlib inline
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")

import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt

## Basic usage

In [2]:
from spice.models import IcosphereModel
from spice.spectrum import simulate_observed_flux, Blackbody

bb = Blackbody()
star = IcosphereModel.construct(1000, 1., 1., bb.solar_parameters, bb.parameter_names)

wavelengths = np.linspace(4000., 7000., 2000)   # Angstroms
flux = simulate_observed_flux(bb.intensity, star, np.log10(wavelengths))

print("shape:", flux.shape, "| peak flux:", float(flux[:, 0].max()), "erg/s/cm^2/A at 10 pc")

[spice] IcosphereModel constructed in 1.1 s
shape: (2000, 2) | peak flux: 4.211372570117611e-11 erg/s/cm^2/A at 10 pc


## Doppler broadening from rotation

An approaching limb contributes blueshifted light, a receding limb redshifted
light — a rotating star's lines broaden. The blackbody continuum is smooth,
so to *see* the effect we use a Gaussian line-profile emulator.

In [3]:
from spice.models.mesh_transform import add_rotation, evaluate_rotation
from spice.spectrum import GaussianLineEmulator

line_center = 5500.0
gle = GaussianLineEmulator(line_centers=[line_center], line_widths=[0.5], line_depths=[0.5])

line_wavelengths = np.linspace(line_center - 5, line_center + 5, 1000)
log_wl = np.log10(line_wavelengths)

static_flux = simulate_observed_flux(gle.intensity, star, log_wl)

fast = evaluate_rotation(add_rotation(star, rotation_velocity=50.), 0.)
rot_flux = simulate_observed_flux(gle.intensity, fast, log_wl)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(line_wavelengths, static_flux[:, 0] / static_flux[:, 1], label="No rotation")
ax.plot(line_wavelengths, rot_flux[:, 0] / rot_flux[:, 1], label="50 km/s")
ax.set_xlabel(r"Wavelength [$\AA$]")
ax.set_ylabel("Normalized flux")
ax.legend()
plt.show()

/var/folders/7r/n_x0ntj511v_0gt816mgrc1c0000gq/T/ipykernel_33098/1918838101.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Distance scaling

Observed flux scales as $1/d^2$.

In [4]:
flux_10pc = simulate_observed_flux(bb.intensity, star, np.log10(wavelengths), distance=10.)
flux_100pc = simulate_observed_flux(bb.intensity, star, np.log10(wavelengths), distance=100.)

ratio = float(jnp.median(flux_10pc[:, 0] / flux_100pc[:, 0]))
print(f"flux(10 pc) / flux(100 pc) = {ratio:.1f} (expected 100)")

flux(10 pc) / flux(100 pc) = 100.0 (expected 100)


## Luminosities

Note: these are physically meaningful only when the emulator's `flux`
channel is a true surface flux. `Blackbody.flux` returns the mu=1 intensity,
so the values below serve resolution-convergence comparisons, not absolute
photometry (compare the Stefan-Boltzmann calculation in the Synthetic
Photometry page).

In [5]:
from spice.spectrum import simulate_monochromatic_luminosity, luminosity, absolute_bol_luminosity

lum_wavelengths = np.linspace(200., 40000., 20000)

# L_lambda(lambda): (n_wavelengths, 2), erg/s/Angstrom
mono_lum = simulate_monochromatic_luminosity(bb.flux, star, np.log10(lum_wavelengths))

# Bolometric luminosity in erg/s (trapezoidal integral over wavelength)
L = luminosity(bb.flux, star, lum_wavelengths)

# Absolute bolometric magnitude
M_bol = absolute_bol_luminosity(L)

print(f"L = {float(L):.3e} erg/s, M_bol = {float(M_bol):.2f}")

L = 1.212e+25 erg/s, M_bol = 25.99
